# **Pruning with a modified Cambrige-Aachen algorithm**

In order to perform the PDF pruning only based on the PDFs shapes we use the $k_t$-like algorithm with $p=0$, which implies that the algoritm is based only on the geometric information [doi: 10.1007/s100520050232.]. To account the mutual distance between any two PDF replicas (vectors) $\mathbf{v}_i$ and $\mathbf{v}_j$, we define a Dissimilarity  metric as the absolute difference normalized by the average of the absolute values between each vector (This metric is also know as the Symmetric mean absolute percentage error (SMAPE)), i.e.,

$$
    D(\mathbf{v}_i, \mathbf{v}_j) =
\begin{cases} 
\sum_{k=1}^{N} 2 \dfrac{|v_{i,k} - v_{j,k}|}{|v_{i,k}| + |v_{j,k}|} & \text{if } |v_{i,k}| + |v_{j,k}| > \epsilon \\
0 & \text{if } |v_{i,k}| + |v_{i,j}| \leq \epsilon
\end{cases}
$$

where $\epsilon$ is a tolerance value (in our case $\epsilon=0.0001$). This tolerance value defines when a difference between PDFs is acceptable or is taken as a numerical noise, it ensures numerical stability and focus on physically significant regions

Then, the Cambrige-Aachen algorithm to find similar PDFs becomes,

$$
    d_{ij}=\dfrac{D(\mathbf{v}_i, \mathbf{v}_j)^2}{R^2}, ~~ d_{B}=1
$$

where $R$ is a hyperparemter that defines the set size of the PDFs that are consider similar. Also, the recombination scheme is taken as
$$
    \mathbf{v}_{new}=\dfrac{ \mathbf{v}_i+ \mathbf{v}_j }{2}
$$



# Packages

In [1]:
import pruning_tools as pt
import pandas as pd
from pathlib import Path

#  1.0 Loading Data

In [2]:
# Loading vectors
data_folder = Path("260131_data")
vec_filename = "260131.PN.ct25alt_nx21.tsv"

df_vectors = pd.read_csv(data_folder / vec_filename, sep='\t', header=None)

df_vectors

,0,1,2,3,4,5,6,7,8,9,...,158,159,160,161,162,163,164,165,166,167
0,726.924083,463.697216,300.426248,197.572658,131.204079,87.563117,58.482450,38.929631,25.735483,16.844045,...,17.727170,11.614090,7.496735,4.711759,2.844302,1.611816,0.830573,0.370840,0.134039,0.036553
1,724.493802,462.582262,299.885426,197.273306,131.082706,87.506601,58.463031,38.929631,25.741889,16.852019,...,17.848767,11.692118,7.539306,4.733358,2.851145,1.615590,0.833063,0.372767,0.134596,0.035519
2,710.542188,458.741863,299.164331,197.472874,131.406367,87.789184,58.647518,39.032405,25.789939,16.865974,...,17.848767,11.734133,7.586134,4.761129,2.860268,1.613531,0.829263,0.371651,0.135153,0.035279
3,714.232615,460.847888,300.005609,197.672442,131.365910,87.694989,58.569839,38.983722,25.777126,16.875942,...,17.258151,11.494048,7.539306,4.791985,2.896763,1.626397,0.819831,0.355519,0.126801,0.037033
4,725.483917,463.325565,300.366156,197.672442,131.284995,87.638473,58.531000,38.962086,25.757906,16.856006,...,17.814025,11.710124,7.569106,4.751872,2.857988,1.611473,0.827298,0.369521,0.133880,0.035412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,1045.920982,508.667050,297.211365,189.889277,126.753740,86.131362,58.637808,39.573319,26.299267,17.153055,...,17.631629,11.692118,7.628705,4.804328,2.857988,1.567386,0.774503,0.339793,0.131653,0.045105
282,1096.326812,537.036450,309.770439,194.080212,126.996486,85.038707,57.443496,38.772766,25.940495,17.105208,...,17.666371,11.728130,7.654248,4.819756,2.862549,1.563269,0.768215,0.334922,0.129107,0.044222
283,1012.617129,511.888030,301.658119,191.785176,126.794198,85.490840,57.967828,39.173042,26.190354,17.236787,...,17.683742,11.710124,7.590391,4.745701,2.826055,1.573219,0.799525,0.358867,0.134835,0.038811
284,1012.617129,511.888030,301.658119,191.785176,126.794198,85.490840,57.967828,39.173042,26.193558,17.236787,...,17.683742,11.710124,7.590391,4.745701,2.826055,1.573047,0.799525,0.358867,0.134835,0.038811


In [3]:
met_filename = "260131.PN.ct25alt_nx21_metadata.tsv"
metadata = pd.read_csv(data_folder / met_filename, sep="\t", usecols=[0,1])
metadata

,name,chi2f
0,pq125a003,4949
1,pq125a004,4973
2,pq125a005,4945
3,pq125a006,4948
4,pq125a007,4971
...,...,...
281,pq125a284,5019
282,pq125a285,5019
283,pq125a286,5019
284,pq125a287,5054


# 2.0 **Clustering Analisis**

## 2.1 Neighborhood Degree and Pairwise Distance Analysis

The $R$ parameter is the only hyperparameter on this algoritm and it defines the set size of the PDFs that are consider similar. Therefore for pruning purposes we should take $R$ enough to include in a same family only the replicas that are really close.

To determine an appropriate distance threshold empirically, we computed all pairwise PDF dissimilarities for the vector set with $n$ replicas. The resulting figure shows the Cumulative Distribution Function (CDF), which represents the fraction of replica pairs separated by a distance less than or equal to a given value $R$.

In [15]:
vec = df_vectors.values
plt_cdf = pt.plot_pairwise_cdf(data_input=vec, R_cutoff=3, metric=pt.pdf_dissimilarity)
plt_cdf.show()  

────────────────────────────────────────────────────
  Pairwise Distance Statistics  (Custom Metric)
────────────────────────────────────────────────────
  Vectors (N)        : 286
  Pairs  N(N-1)/2    : 40,755

  Min distance       : 0.0028
  Mean distance      : 10.5934
  Median distance    : 10.6609
  Std deviation      : 4.0053
  Max distance       : 23.9386

  Cutoff  R = 3
    R is 1.91 std devs below the median

  Pairs with d ≤ R   : 1,095 / 40,755
  → Merged (pruned)  : 2.69%   [CDF(3) = 0.0269]
  → Unmerged         : 97.31%
────────────────────────────────────────────────────


In this case R=2 will merge only vectors that are 1% 

In [16]:
pt.plot_neighbor_degree(data_input=df_vectors, R_cutoff=2.0, metric=pt.pdf_dissimilarity )

────────────────────────────────────────────────────
  Neighbor degree  (R = 2.0, Custom Metric)
────────────────────────────────────────────────────
  N vectors                : 286
  Isolated  (k=0)          : 54  (18.9%)
    → guaranteed solo clusters
  With neighbors (k≥1)     : 232  (81.1%)
  Mean degree              : 2.92
  Median degree            : 2.0
  Max degree (biggest hub) : 11
  Total close pairs        : 417
────────────────────────────────────────────────────


In [17]:
pt.neighbor_degree_analysis(data_input=df_vectors, R_cutoff=2.0, metric=pt.pdf_dissimilarity)

────────────────────────────────────────────────────
  Neighbor degree analysis  (R = 2.0)
────────────────────────────────────────────────────
  Vectors (N)              : 286

  Isolated  (0 neighbors)  : 54
  → These will ALWAYS form solo clusters, regardless of R.

  With >= 1 neighbor       : 232
  → These are candidates for merging.

  Min neighbors            : 0
  Max neighbors            : 11  ← hub size
  Mean neighbors           : 2.92
  Median neighbors         : 2.0

  Vectors above mean       : 131  ← potential cluster cores

  Pairs check (sum/2)      : 417
  → Should match n_pairs_below_cutoff from plot_pairwise_cdf.

  Predicted clusters (rough estimate)
  → Lower bound (aggressive): 54 isolated + 131 hubs = ~185
  → Upper bound              = N = 286  (no merging at all)
────────────────────────────────────────────────────


{'counts': array([ 2,  2,  4,  2,  0,  3,  1,  4,  1,  4,  3,  1,  3,  2,  2,  0,  2,
         1,  0,  4,  2,  6,  2,  4,  1,  3,  1,  0,  2,  4,  1,  0,  1,  0,
         4,  6,  3,  1,  2,  1,  7,  7,  3,  0,  1,  5,  0,  0,  0,  3,  5,
         2,  7,  5,  4,  0,  2,  0,  2,  1,  0,  4,  1,  2,  3,  1,  4,  2,
         0,  0,  1,  4,  2,  2,  4,  1,  5,  2,  1,  1,  2,  2,  2,  2,  1,
         7,  3,  2,  7,  7,  3,  0,  0,  0,  3,  0,  0,  1,  2,  7,  1,  3,
         1,  0,  3,  3,  1,  7,  2,  4,  1,  2,  1,  0,  1,  2,  2,  0,  2,
         6,  2,  1,  0,  4,  3,  6,  0,  1,  1,  3,  2,  7,  5,  7,  7,  4,
         8,  5,  0,  5,  3,  3,  5,  5,  5,  8,  1,  7, 10,  5,  8,  5,  8,
         8,  0,  2,  7,  6,  7,  6,  2,  1,  0,  2,  2,  4,  0,  0,  3,  2,
         1,  0,  9,  0,  3,  0,  1,  2,  4,  3,  4,  7,  3,  2,  3,  1,  1,
         0,  0,  4,  0,  0,  5,  2,  4,  4,  0,  0,  0,  3,  1,  0,  2,  0,
         4,  3,  2,  3,  3,  1,  2,  2,  2,  0,  2,  9,  7,  3,  7,  2,  1,
  

## 2.2 Clustering search

In [6]:
#clustering serach
vec = df_vectors.values
clstr = pt.cambridge_cluster(vectors=vec, R=2.0, metric=pt.pdf_dissimilarity)


In [7]:
clstr_results = pd.DataFrame(clstr).sort_values(by='num_vectors', ascending=False)
clstr_results

,cluster_label,vectors_in_cluster,cluster_vector,num_vectors
100,100,"[215, 225, 222, 230, 216, 223, 268, 270, 269, ...","[1048.08685709375, 516.4678609, 301.3632958781...",15
52,52,"[99, 159, 134, 150, 153, 152, 131, 145, 147, 1...","[672.096491421875, 478.1896897859375, 308.7094...",14
87,87,"[189, 195, 192, 194, 253, 272, 259, 271, 254, ...","[834.0477244, 472.43102689375, 295.49313104375...",10
19,19,"[35, 40, 41, 45, 52, 89, 125, 107, 85]","[930.50513158125, 501.41597426875, 304.3246682...",9
66,66,"[139, 140, 151, 149, 156, 158, 172, 181, 157]","[629.7606569187501, 454.23945913750003, 299.63...",9
...,...,...,...,...
112,112,[243],"[1049.521398, 522.2942728, 304.6626815, 192.28...",1
111,111,[242],"[1027.018795, 515.9761969, 302.8599438, 191.98...",1
119,119,[262],"[1027.018795, 522.7898082, 310.0708948, 196.97...",1
120,120,[278],"[1382.559921, 627.1000021, 342.5201745, 205.75...",1


## 2.3 New metadata, best and discard vectors

In [8]:
# New Metadata
new_metadata = pt.assign_cluster_data_to_metadata(metadata_df=metadata, final_jets=clstr, metric_column="chi2f")
new_metadata

,name,chi2f,cluster_label,cluster_size,cluster_best_idx,cluster_best_chi2f,cluster_avg_chi2f
0,pq125a003,4949,0,4,0,4949,4958.75
1,pq125a004,4973,0,4,0,4949,4958.75
2,pq125a005,4945,1,5,2,4945,4969.40
3,pq125a006,4948,2,6,3,4948,4980.00
4,pq125a007,4971,3,1,4,4971,4971.00
...,...,...,...,...,...,...,...
281,pq125a284,5019,121,2,281,5019,5019.00
282,pq125a285,5019,121,2,281,5019,5019.00
283,pq125a286,5019,122,2,283,5019,5036.50
284,pq125a287,5054,122,2,283,5019,5036.50


In [9]:
best_vectors, discarded_vectors = pt.get_pruned_and_discarded_vectors(df_vectors, new_metadata, "chi2f")

--- Pruning Summary (chi2f) ---
Total Original : 286
Kept (Best)    : 124
Discarded      : 162
Reduction      : 56.6%


In [10]:
best_vectors.to_csv(data_folder / (vec_filename + "_best_replicas.tsv"), sep="\t", index=False)
best_vectors

,0,1,2,3,4,5,6,7,8,9,...,159,160,161,162,163,164,165,166,167,chi2f
0,726.924083,463.697216,300.426248,197.572658,131.204079,87.563117,58.482450,38.929631,25.735483,16.844045,...,11.614090,7.496735,4.711759,2.844302,1.611816,0.830573,0.370840,0.134039,0.036553,4949
2,710.542188,458.741863,299.164331,197.472874,131.406367,87.789184,58.647518,39.032405,25.789939,16.865974,...,11.734133,7.586134,4.761129,2.860268,1.613531,0.829263,0.371651,0.135153,0.035279,4945
3,714.232615,460.847888,300.005609,197.672442,131.365910,87.694989,58.569839,38.983722,25.777126,16.875942,...,11.494048,7.539306,4.791985,2.896763,1.626397,0.819831,0.355519,0.126801,0.037033,4948
4,725.483917,463.325565,300.366156,197.672442,131.284995,87.638473,58.531000,38.962086,25.757906,16.856006,...,11.710124,7.569106,4.751872,2.857988,1.611473,0.827298,0.369521,0.133880,0.035412,4971
6,712.432407,461.591191,301.057206,198.670284,132.013232,88.071767,58.764036,39.064860,25.786736,16.854013,...,11.626095,7.518021,4.721016,2.842021,1.607870,0.831491,0.377130,0.140085,0.037764,4965
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267,973.012548,486.615727,289.730004,187.594241,126.349164,86.376268,58.987363,39.854595,26.475450,17.248748,...,11.668109,7.637219,4.804328,2.844302,1.552977,0.768084,0.339691,0.133085,0.045559,5001
278,1382.559921,627.100002,342.520174,205.754960,130.314011,85.095224,56.491930,37.734209,25.168495,16.652658,...,11.824164,7.718104,4.850612,2.880797,1.583340,0.786556,0.343750,0.127835,0.038717,5068
281,1045.920982,508.667050,297.211365,189.889277,126.753740,86.131362,58.637808,39.573319,26.299267,17.153055,...,11.692118,7.628705,4.804328,2.857988,1.567386,0.774503,0.339793,0.131653,0.045105,5019
283,1012.617129,511.888030,301.658119,191.785176,126.794198,85.490840,57.967828,39.173042,26.190354,17.236787,...,11.710124,7.590391,4.745701,2.826055,1.573219,0.799525,0.358867,0.134835,0.038811,5019


In [11]:
discarded_vectors.to_csv( data_folder / (vec_filename + "_discarded_replicas.tsv"), sep="\t", index=False)
discarded_vectors

,0,1,2,3,4,5,6,7,8,9,...,158,159,160,161,162,163,164,165,166,167
1,724.493802,462.582262,299.885426,197.273306,131.082706,87.506601,58.463031,38.929631,25.741889,16.852019,...,17.848767,11.692118,7.539306,4.733358,2.851145,1.615590,0.833063,0.372767,0.134596,0.035519
5,711.892345,459.485165,299.584970,197.772226,131.568198,87.902217,58.705777,39.059450,25.799549,16.867968,...,17.822710,11.722128,7.581877,4.761129,2.862549,1.614389,0.829263,0.371144,0.134517,0.035002
7,717.743021,462.086726,300.456293,197.872011,131.406367,87.676151,58.521290,38.935040,25.725873,16.830089,...,17.701113,11.632097,7.513764,4.717930,2.835178,1.602038,0.827691,0.375710,0.140165,0.038212
8,714.412636,461.095656,300.245974,197.872011,131.487283,87.751506,58.569839,38.962086,25.738686,16.834077,...,17.709799,11.632097,7.518021,4.717930,2.837459,1.603410,0.828346,0.375710,0.139847,0.037979
9,730.074448,466.670428,302.259031,198.470716,131.608655,87.732667,58.550419,38.972904,25.770719,16.877936,...,17.666371,11.728130,7.637219,4.813584,2.885359,1.612845,0.815115,0.356635,0.125926,0.031647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277,1054.922023,525.019717,306.164963,193.182155,127.117859,85.358968,57.754211,38.978313,26.052611,17.147074,...,17.657685,11.806158,7.709590,4.822841,2.839740,1.541826,0.760224,0.336850,0.133403,0.046739
279,1097.226916,537.531985,310.070895,194.379565,127.239232,85.170579,57.530885,38.821448,25.969325,17.117170,...,17.614258,11.758141,7.688304,4.829013,2.848864,1.544400,0.754590,0.328936,0.127914,0.044695
280,1052.221711,511.020843,298.172826,190.388198,126.956029,86.225557,58.676648,39.594956,26.308877,17.159036,...,17.579516,11.728130,7.667019,4.816670,2.846583,1.548688,0.760093,0.332995,0.130062,0.045446
282,1096.326812,537.036450,309.770439,194.080212,126.996486,85.038707,57.443496,38.772766,25.940495,17.105208,...,17.666371,11.728130,7.654248,4.819756,2.862549,1.563269,0.768215,0.334922,0.129107,0.044222


## 2.4 Cluster serparation

$$\text{Separation Ratio}_i = \frac{\text{Avg\_Inter\_Dist}_i}{\text{Avg\_Intra\_Dist}_i + \epsilon}$$

Where:
 * $\text{Avg\_Intra\_Dist}_i$ (Tightness): The average distance from the members of cluster $i$ to their representative centroid.
 * $\text{Avg\_Inter\_Dist}_i$ 
 (Separation): The average distance from the centroid of cluster $i$ to the centroids of all other clusters.
 * $\epsilon$ ($1e-9$): A tiny constant added to the denominator to prevent "Division by Zero" errors if a cluster contains only one vector.


Ideally, intra-cluster distances should be small while inter-cluster distances are large, indicating well-separated, cohesive clusters.

In [12]:
# Distances beteween the centroid and the internal elements of the cluster (Intra_Dist)
# Distances between each centroid of each cluster (Inter_Dist)
cluster_sep=pt.analyze_cluster_separation(clstr, df_vectors)
cluster_sep

,Cluster,Size,Avg_Intra_Dist,Avg_Inter_Dist,Separation_Ratio
0,0,4,1443.888720,7465.184575,5.170194e+00
1,1,5,661.590900,7584.209605,1.146359e+01
2,2,6,870.836030,8656.712955,9.940692e+00
3,3,1,0.000000,7428.473528,7.428474e+12
4,4,2,43.426539,7192.541986,1.656255e+02
...,...,...,...,...,...
119,119,1,0.000000,9383.254688,9.383255e+12
120,120,1,0.000000,8827.159156,8.827159e+12
121,121,2,51.570703,8317.713629,1.612876e+02
122,122,2,4.543898,8302.975370,1.827280e+03


In [13]:
pltcluster_sep = pt.plot_cluster_separation(cluster_sep)
pltcluster_sep.show()